# verify_book1 — LLaVA 13B (Local Ollama)

Sends each question from `book1.json` to **llava:13b** running via Ollama inside Colab.

- Questions + sub-questions are combined into one prompt
- Images are embedded as base64 in the JSON — no extra uploads needed
- Answers are written to `/content/outputs/book1/<id>.txt`
- Already-answered entries are skipped on re-run

> **Runtime:** GPU (T4 or better) recommended.

In [ ]:
# Install Ollama Python client
!pip install -q ollama

In [ ]:
# Install and start Ollama service, then pull llava:13b
import subprocess, time, os

!curl -fsSL https://ollama.com/install.sh | sh

proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print("Ollama service started.")

!ollama pull llava:13b
print("Model ready.")

In [ ]:
# Upload book1.json
from google.colab import files
uploaded = files.upload()  # select book1.json
JSON_PATH = list(uploaded.keys())[0]
print("Using:", JSON_PATH)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL           = "llava:13b"
OUTPUT_DIR      = "/content/outputs/book1"
POST_CALL_DELAY = 3   # seconds between calls

SYSTEM_PROMPT = (
    "You are an expert computer science and networking tutor. "
    "Answer each sub-question step by step. "
    "Label your answers clearly (e.g. (a), (b), (c)). "
    "Show your reasoning before giving the final answer."
)

# Filter: set to None to run all, or e.g. ["1", "3"] for specific ids
FILTER_IDS  = None
# Filter: set to an integer to run only the last N entries
LAST_N      = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json, time
from ollama import Client

client = Client(host="http://localhost:11434")

def load_entries(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def build_prompt(entry):
    lines = [entry["question"].strip()]
    sub_qs = entry.get("sub_questions", [])
    if sub_qs:
        lines.append("")
        lines.append("Sub-questions:")
        for sq in sub_qs:
            lines.append(f"  ({sq.get('id', '')}) {sq.get('sub_question', '').strip()}")
    return "\n".join(lines)

def ask(entry):
    out_path = os.path.join(OUTPUT_DIR, f"{entry['id']}.txt")
    if os.path.exists(out_path):
        print(f"  [SKIP] #{entry['id']} — already answered")
        return

    prompt = build_prompt(entry)
    if not prompt.strip():
        print(f"  [SKIP] #{entry['id']} — empty question")
        return

    images = [img["base64"] for img in entry.get("images", []) if img.get("base64")]

    user_msg = {"role": "user", "content": prompt}
    if images:
        user_msg["images"] = images

    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            user_msg,
        ],
        options={"temperature": 0},
    )
    answer = response["message"]["content"].strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(answer)
    print(f"  [OK]   #{entry['id']} → {out_path}")

print("Helpers defined.")

In [ ]:
# Load and filter entries
all_entries = load_entries(JSON_PATH)

if FILTER_IDS:
    id_set  = set(str(i) for i in FILTER_IDS)
    entries = [e for e in all_entries if str(e["id"]) in id_set]
elif LAST_N:
    entries = all_entries[-LAST_N:]
else:
    entries = all_entries

print(f"Model  : {MODEL}")
print(f"Output : {OUTPUT_DIR}")
print(f"Running {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}…")

In [ ]:
# Run
for i, entry in enumerate(entries):
    try:
        ask(entry)
    except Exception as exc:
        print(f"  [ERR]  #{entry['id']} — {exc}")
    if i < len(entries) - 1:
        time.sleep(POST_CALL_DELAY)

print("\nDone.")

In [ ]:
# Download all output files as a zip
import shutil
from google.colab import files

zip_path = "/content/book1_llava_answers"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")